# eQTL-Style Multi-Trait LOCO MLM Acceleration

This notebook shows a practical eQTL/QTL pattern: many traits (genes) tested against the same genotype matrix and the same LOCO kinship.

It demonstrates:
1. Baseline per-trait LOCO MLM (`PANICLE_MLM_LOCO` called once per trait)
2. Accelerated multi-trait LOCO MLM (`PANICLE_MLM_LOCO_MULTI`)
3. Writing QTL hits to a CSV output table

The acceleration comes from processing chromosome-major and reusing the expensive genotype eigenspace transform (`U'G`) across traits.

## Imports

In [ ]:
import time
import numpy as np
import pandas as pd

from panicle.utils.data_types import GenotypeMatrix
from panicle.matrix.kinship_loco import PANICLE_K_VanRaden_LOCO
from panicle.association.mlm_loco import PANICLE_MLM_LOCO, PANICLE_MLM_LOCO_MULTI

## Build Synthetic eQTL-Style Data

- Same individuals and genotypes for all traits
- Multiple chromosomes in map data
- A few traits with simulated marker effects

In [ ]:
rng = np.random.default_rng(2026)

n_individuals = 180
n_markers = 2400
n_traits = 6

geno = rng.integers(0, 3, size=(n_individuals, n_markers), dtype=np.int8)
geno_matrix = GenotypeMatrix(geno, is_imputed=True)

chrom_labels = np.array([str(i) for i in range(1, 7)])
chrom = np.repeat(chrom_labels, n_markers // chrom_labels.size)
if chrom.size < n_markers:
    chrom = np.concatenate([chrom, np.repeat(chrom_labels[-1], n_markers - chrom.size)])

map_df = pd.DataFrame({
    "MARKER": [f"M{i:05d}" for i in range(n_markers)],
    "CHROM": chrom,
    "POS": np.arange(1, n_markers + 1) * 100,
})

trait_names = [f"Gene_{i+1}" for i in range(n_traits)]
signal_markers = [50, 420, 850, 1210, 1750, 2200]

phe_matrix = np.zeros((n_individuals, n_traits), dtype=np.float64)
for t in range(n_traits):
    effect = 0.45 * geno[:, signal_markers[t]].astype(np.float64)
    noise = rng.normal(scale=1.0, size=n_individuals)
    phe_matrix[:, t] = effect + noise

covariates = rng.normal(size=(n_individuals, 2))

print(f"Individuals: {n_individuals}")
print(f"Markers: {n_markers}")
print(f"Traits: {n_traits}")

## Precompute LOCO Kinship (Shared Across Traits)

In [ ]:
loco = PANICLE_K_VanRaden_LOCO(
    geno_matrix,
    map_df,
    maxLine=512,
    cpu=1,
    verbose=False,
)

## Baseline: Run One Trait at a Time

In [ ]:
single_trait_results = {}
t0 = time.perf_counter()

for idx, trait_name in enumerate(trait_names):
    phe_single = np.column_stack([np.arange(n_individuals), phe_matrix[:, idx]])
    single_trait_results[trait_name] = PANICLE_MLM_LOCO(
        phe=phe_single,
        geno=geno_matrix,
        map_data=map_df,
        loco_kinship=loco,
        CV=covariates,
        maxLine=256,
        cpu=1,
        lrt_refinement=False,
        verbose=False,
    )

single_time = time.perf_counter() - t0
print(f"Per-trait loop runtime: {single_time:.2f} seconds")

## Accelerated: Run Traits Together (Chromosome-Major)

`PANICLE_MLM_LOCO_MULTI` processes one chromosome at a time, computes `U'G_chrom` once, runs all matching traits, then releases that matrix.

In [ ]:
t0 = time.perf_counter()

multi_results = PANICLE_MLM_LOCO_MULTI(
    phe=phe_matrix,
    geno=geno_matrix,
    map_data=map_df,
    trait_names=trait_names,
    loco_kinship=loco,
    CV=covariates,
    maxLine=256,
    cpu=1,
    lrt_refinement=False,
    verbose=False,
)

multi_time = time.perf_counter() - t0
print(f"Grouped multi-trait runtime: {multi_time:.2f} seconds")
print(f"Speedup (single/multi): {single_time / multi_time:.2f}x")

## Numerical Comparison

In [ ]:
rows = []
for trait_name in trait_names:
    a = single_trait_results[trait_name]
    b = multi_results[trait_name]
    rows.append({
        "trait": trait_name,
        "max_abs_effect_diff": float(np.max(np.abs(a.effects - b.effects))),
        "max_abs_se_diff": float(np.max(np.abs(a.se - b.se))),
        "max_abs_p_diff": float(np.max(np.abs(a.pvalues - b.pvalues))),
    })

pd.DataFrame(rows)

## Write QTL Hit Table

This writes a compact long-format table with top hits per trait.

In [ ]:
top_k = 10
all_hits = []

marker_values = map_df["MARKER"].to_numpy()
chrom_values = map_df["CHROM"].to_numpy()
pos_values = map_df["POS"].to_numpy()

for trait_name in trait_names:
    res = multi_results[trait_name]
    top_idx = np.argpartition(res.pvalues, top_k - 1)[:top_k]
    top_idx = top_idx[np.argsort(res.pvalues[top_idx])]

    hit_df = pd.DataFrame({
        "Trait": trait_name,
        "MARKER": marker_values[top_idx],
        "CHROM": chrom_values[top_idx],
        "POS": pos_values[top_idx],
        "P_VALUE": res.pvalues[top_idx],
        "EFFECT": res.effects[top_idx],
        "SE": res.se[top_idx],
    })
    all_hits.append(hit_df)

qtl_hits = pd.concat(all_hits, ignore_index=True)
output_path = "eqtl_multitrait_top_hits.csv"
qtl_hits.to_csv(output_path, index=False)

print(f"Wrote: {output_path}")
qtl_hits.head(20)

## Notes

- Real speedups are largest when many traits share the same sample set and the same LOCO kinship.
- This notebook uses `lrt_refinement=False` to isolate the Wald-path acceleration.
- For production analyses you can enable LRT refinement when needed for top-hit calibration.